In [ ]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm


In [ ]:
def flatten_data(folder_path):
    data_dir = Path(folder_path)
    final_data = []

    
    for i in tqdm(list(data_dir.glob('*.json'))):
        with i.open('r') as f:
            match_data = json.load(f)

        
        info = match_data.get('info', {})
        if info.get('gender') != 'male' or info.get('match_type') != 'T20':
            continue

       
        venue = info.get('venue', 'Unknown')
        
       
        outcome = info.get('outcome', {})
        winner = outcome.get('winner')

        toss = info.get('toss', {})
        toss_winner = toss.get('winner')
        

        match_date = info.get('dates', ['1970-01-01'])[0]  
        match_id = i.stem

        
        if len(match_data.get('innings', [])) < 2:
            continue
            
        second_innings = match_data['innings'][1]
        
       
        target = second_innings.get('target', {}).get('runs', 0)
        if target == 0: 
            continue 

        batting_team = second_innings.get('team')
        
       
        teams = info.get('teams', [])
        bowling_team_list = [t for t in teams if t != batting_team]
        if not bowling_team_list:
            continue
        bowling_team = bowling_team_list[0]

        
        is_chaser_winner = 1 if winner == batting_team else 0

        chasing_team_toss_winner=1 if toss_winner==batting_team else 0


        current_score = 0
        wickets_lost = 0

        
        for over in second_innings.get('overs', []):
            over_num = over.get('over', 0)
            
            for ball_index, delivery in enumerate(over.get('deliveries', [])):
                
                runs = delivery.get('runs', {}).get('total', 0)
                current_score += runs
                
                if 'wickets' in delivery:
                    wickets_lost += len(delivery['wickets'])

                
                runs_left = target - current_score
                balls_delivered = (over_num * 6) + (ball_index + 1)
                balls_left = 120 - balls_delivered
                wickets_left = 10 - wickets_lost
                
                
                if balls_left < 0: 
                    balls_left = 0

                
                crr = (current_score * 6) / balls_delivered if balls_delivered > 0 else 0
                rrr = (runs_left * 6) / balls_left if balls_left > 0 else 0

                striker = delivery.get('batter')
                non_striker = delivery.get('non_striker')
                current_bowler = delivery.get('bowler')

                final_data.append({
                    'batting_team': batting_team,
                    'bowling_team': bowling_team,
                    'venue': venue,
                    'runs_left': runs_left,
                    'balls_left': balls_left,
                    'wickets_left': wickets_left,
                    'target_runs': target,
                    'crr': crr,
                    'rrr': rrr,
                    'result': is_chaser_winner,
                    'striker': striker,              
                    'non_striker': non_striker,      
                    'bowler': current_bowler,
                    'chasing_team_toss_winner':chasing_team_toss_winner,
                    "match_id":match_id,
                    'date':match_date
                })

    return pd.DataFrame(final_data)

In [ ]:
df=flatten_data("D:\Semester 4\AI\Lab\Project\\t20s_json")

In [ ]:
#removing the rows where the match is already won
df=df[df['runs_left']>=0]

#removing rain affected or unrealistic matches
df=df[df['target_runs']>=50]

#this will avoid infinite rrr error
df=df[df['balls_left']>0]

In [ ]:
import pandas as pd


match_df = df[['match_id', 'date', 'batting_team', 'bowling_team', 'result']].drop_duplicates()


match_df['matchup'] = match_df.apply(lambda x: tuple(sorted([x['batting_team'], x['bowling_team']])), axis=1)


match_df = match_df.sort_values('date')

def get_last_5_wins(row, all_matches):
    
    past_matches = all_matches[
        (all_matches['matchup'] == row['matchup']) & 
        (all_matches['date'] < row['date'])
    ].tail(5)
    
    if past_matches.empty:
        return 0 
        
   
    wins = 0
    for _, match in past_matches.iterrows():
      
        if match['batting_team'] == row['batting_team'] and match['result'] == 1:
            wins += 1
        
        elif match['bowling_team'] == row['batting_team'] and match['result'] == 0:
            wins += 1
            
    return wins



match_df['chasing_team_last_5_wins'] = match_df.apply(lambda x: get_last_5_wins(x, match_df), axis=1)


df = df.merge(match_df[['match_id', 'chasing_team_last_5_wins']], on='match_id', how='left')


In [ ]:
df.describe()

In [ ]:
#converting to csv file
df.to_csv('processed_data.csv',index=False)

In [ ]:
df.head()